# HW3 — Part 2: Large FrozenLake with Optimistic Exploration
### REINFORCE · Exploration Bonus β/√count · Sparse Rewards
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## Ice Age — 10×10 FrozenLake with Optimistic Exploration

**Problem:** Standard REINFORCE fails on this 10×10 grid — a random agent almost never reaches the goal (sparse reward).

**Solution — Exploration Bonus:**  
`augmented_reward = real_reward + β / √(visit_count(state))`

- Rare states → high bonus (optimistic value estimate)  
- Frequent states → tiny bonus (bonus shrinks naturally)
- β = 0.1 (selected hyperparameter)

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F, gymnasium as gym, matplotlib.pyplot as plt

DEVICE=torch.device("cpu"); BETA=0.1; GAMMA=0.99; LR=0.01; EPISODES=25_000

MAP=[
    'SHFFFFFHFF','FFFFFFFFHH','FFFHHHFFFF','FFFFFFFFHF','FFFFFFHFHF',
    'FFFFFFFFFH','FFFFFHHFHF','FHFFFFFFFF','FFFHFFFFFF','FFFFHFFFHG'
]
env=gym.make('FrozenLake-v1',desc=MAP,is_slippery=True)
ns,na=env.observation_space.n,env.action_space.n

class PolicyNet(nn.Module):
    def __init__(self): super().__init__(); self.logits=nn.Parameter(torch.zeros(ns,na))
    def forward(self,s): return F.softmax(self.logits[s],dim=-1)

policy=PolicyNet().to(DEVICE)
opt=optim.Adam(policy.parameters(),lr=LR)
counts=np.ones(ns); actual_returns=[]

print(f"Training REINFORCE + Optimistic Exploration (β={BETA}, {EPISODES:,} episodes)...")
for ep in range(EPISODES):
    s,_=env.reset(); lps,rs_bonus,rs_real=[],[],[]
    counts[s]+=1; done=False
    while not done:
        pr=policy(s).detach().numpy(); pr=np.clip(pr,1e-8,None); pr/=pr.sum()
        a=np.random.choice(na,p=pr)
        ns_,r,done,_,_=env.step(a)
        bonus=BETA/np.sqrt(counts[s])
        rs_bonus.append(r+bonus); rs_real.append(r)
        lps.append(torch.log(policy(s)[a])); counts[s]+=1; s=ns_
    G=loss=0.0
    for t in range(len(rs_bonus)-1,-1,-1):
        G=rs_bonus[t]+GAMMA*G; loss+=-lps[t]*G
    opt.zero_grad(); loss.backward(); opt.step()
    actual_returns.append(sum(rs_real))
    if (ep+1)%5000==0:
        print(f"  Ep {ep+1:>6} | avg return={np.mean(actual_returns[-5000:]):.5f} | visited {(counts>2).sum()}/{ns} states")

env.close(); print("\n✅ Done.")
w=1000; plt.figure(figsize=(10,4))
plt.plot(np.convolve(actual_returns,np.ones(w)/w,'valid'))
plt.xlabel("Episode"); plt.ylabel("Actual Return")
plt.title(f"Large FrozenLake — REINFORCE + Optimistic Exploration (β={BETA})"); plt.tight_layout(); plt.show()
